# BigQuery: Agentic Migration & Data Transfer (Managed MCP)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_migration_mcp_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_migration_mcp_demo.ipynb)

This notebook shows how an agent can translate legacy SQL and manage data transfers into BigQuery using the new **BQMS** and **DTS** Managed MCP servers — no custom scripts needed.

## Demonstrates
1. BigQuery Migration Service MCP server to perform SQL translation tasks. (Release: [March 25, 2026](https://docs.cloud.google.com/release-notes#March_25_2026))
2. BigQuery Data Transfer Service remote MCP server to enable AI agents to create, manage, and run data transfers. (Release: [March 24, 2026](https://docs.cloud.google.com/release-notes#March_24_2026))


## Use Case
A migration partner needs to move data from Hive-on-GCS to BigQuery. An agent handles the work:
1.  **Translate SQL**: Convert Hive SQL to GoogleSQL using the BQMS MCP server.
2.  **Generate DDL**: Create matching `CREATE TABLE` statements for BigQuery.
3.  **Active Migration**: Create a transfer config and immediately trigger the data load.

### Requirements
- BigQuery Migration Service and Data Transfer Service APIs enabled.
- `google-adk >= 1.28.0` installed.
- Gemini 3.1 Pro (Preview) access.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai google-cloud-bigquery google-cloud-storage nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
import time
import google.auth
from google.auth.transport.requests import Request

nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
service_account = 'YOUR_SERVICE_ACCOUNT' # @param {type:"string"}
#user_email = service_account
location = 'us-central1'  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. Project Configuration & Service Enablement

Both BQMS and DTS Managed MCP servers must be enabled, along with the source/target APIs.

In [ ]:

# 1. Install beta components via apt-get (Standard for Colab/Notebook environments)
!sudo apt-get update -y
!sudo apt-get install google-cloud-cli-beta -y

# 2. Set the default project globally to avoid missing property errors
!gcloud config set project {project_id} --quiet

# 3. Enable services using the standard and beta commands
import shutil

if shutil.which("gcloud"):
    print("Enabling standard GCP APIs...")
    !gcloud services enable bigquery.googleapis.com bigquerymigration.googleapis.com bigquerydatatransfer.googleapis.com storage.googleapis.com --quiet

    print("Enabling Managed MCP services...")
    !gcloud beta services mcp enable bigquerymigration.googleapis.com --quiet
    !gcloud beta services mcp enable bigquerydatatransfer.googleapis.com --quiet

    print("Success: APIs and Managed MCP services enabled.")
else:
    print("WARNING: gcloud CLI not found. Install: https://cloud.google.com/sdk/docs/install")


### 3. IAM Roles Setup

To use Managed MCP servers, your identity must have the **MCP Tool User** role. We grant the required roles below.

In [ ]:
# Grant required roles to the user, you will need to grant more roles to give access to GCS bucket as well.
# This codeblock may fail if you don't have rights to grant roles to the SA/User

roles = [
    "roles/mcp.toolUser",
    "roles/bigquerymigration.editor",
    "roles/bigquery.admin"
]

for role in roles:
    print(f"Granting {role} to {service_account}...")
    !gcloud projects add-iam-policy-binding {project_id} --member=user:{service_account} --role={role} --condition=None --quiet > /dev/null

print("\nSuccess: IAM Roles configured. Waiting 30s for propagation...")
time.sleep(30)

### 4. Infrastructure Prerequisites

To simulate a Hive-to-BigQuery migration, we create a target dataset, a destination table with the correct schema, and a source bucket with sample 'Hive' data.

In [ ]:
from google.cloud import bigquery, storage

def setup_migration_prereqs():
    # 1. Setup BigQuery Target Dataset
    bq_client = bigquery.Client(project=project_id, location=location)
    dataset_id = f"{project_id}.legacy_migration_target"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location
    bq_client.create_dataset(dataset, exists_ok=True)

    # 2. Setup Destination Table (DTS requires table to exist for GCS loads)
    table_id = f"{dataset_id}.hive_table"
    schema = [
        bigquery.SchemaField("id", "INTEGER"),
        bigquery.SchemaField("first_name", "STRING"),
        bigquery.SchemaField("last_name", "STRING"),
        bigquery.SchemaField("dt", "DATE"),
    ]
    bq_client.delete_table(table_id, not_found_ok=True)
    table = bigquery.Table(table_id, schema=schema)
    bq_client.create_table(table)

    # 3. Setup GCS Source (Simulating Hive External Table storage)
    storage_client = storage.Client(project=project_id)
    bucket_name = f"{project_id}-hive-source"
    bucket = storage_client.create_bucket(bucket_name, location=location) if not storage_client.lookup_bucket(bucket_name) else storage_client.get_bucket(bucket_name)

    # Upload dummy Hive data (CSV format)
    blob = bucket.blob("hive_table/part-0000.csv")
    blob.upload_from_string("1,John,Doe,2025-01-15\n2,Jane,Smith,2025-01-16")

    print(f"Prerequisites ready: Target Table '{table_id}' and Source Bucket 'gs://{bucket_name}/'")

setup_migration_prereqs()

### 5. Initialize Managed MCP Toolsets

We initialize the toolsets for BigQuery Migration (BQMS) and Data Transfer (DTS).

In [ ]:
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams

# Refresh token for MCP
scopes = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=scopes)
creds.refresh(Request())

# 1. Initialize BQMS MCP Toolset
bqms_mcp = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://bigquerymigration.googleapis.com/mcp",
        headers={"Authorization": f"Bearer {creds.token}", "x-goog-user-project": project_id},
        timeout=60.0
    )
)

# 2. Initialize DTS MCP Toolset
dts_mcp = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://bigquerydatatransfer.googleapis.com/mcp",
        headers={"Authorization": f"Bearer {creds.token}", "x-goog-user-project": project_id},
        timeout=60.0
    )
)
print("Managed MCP Toolsets initialized.")

### 6. Define the Migration Assistant Agent

We create a specialized agent equipped with the migration toolsets. We give it explicit instructions to trigger the transfer run.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService

# Gemini 3.1 Pro Preview requires 'global' location
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

migration_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="MigrationAssistant",
    instruction="""
    You are a data migration expert.
    Use BQMS to translate legacy SQL and DTS to manage data transfers into BigQuery.
    Always verify the translation success before suggesting a transfer.

    CRITICAL: If asked to move data, you must first create the transfer config and then
    IMMEDIATELY call 'start_manual_transfer_runs' to trigger the data load now.
    """,
    tools=[bqms_mcp, dts_mcp]
)

runner = Runner(
    agent=migration_agent,
    session_service=InMemorySessionService(),
    app_name="migration_mcp_demo",
    auto_create_session=True
)
print("Migration Assistant Agent and Runner initialized.")

### 7. Run Agentic Migration Workflow

Now we trigger the end-to-end workflow: SQL Translation followed by an **active** Data Transfer execution.

In [ ]:
from google.genai import types
import traceback

async def run_migration_workflow():
    legacy_sql = "SELECT * FROM hive_table WHERE dt > '2025-01-01' LIMIT 10;"
    prompt = f"""Translate this Hive SQL to GoogleSQL: {legacy_sql}
    Then, trigger a BigQuery Data Transfer to move data in CSV file from gs://{project_id}-hive-source/hive_table/
    into the table `legacy_migration_target.hive_table`. Do not just prepare the config; start the transfer run now uning this {service_account} account."""

    print(f"--- Starting Agentic Migration Workflow ---")
    print(f"User Prompt: {prompt}\n")

    try:
        message = types.Content(parts=[types.Part(text=prompt)], role='user')
        async for event in runner.run_async(
            user_id="partner_user",
            session_id="migration_session",
            new_message=message
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Agent: {part.text}")
                    if part.function_call:
                        print(f"[SYSTEM]: {event.author} calling tool '{part.function_call.name}'")
    except Exception as e:
        print(f"\n[ERROR]: Workflow failed.")
        if isinstance(e, BaseExceptionGroup):
            for sub_e in e.exceptions:
                print(f"  Sub-exception: {sub_e}")
        traceback.print_exc()

await run_migration_workflow()

### 8. [VERIFICATION] Confirm Data Arrival

We wait a few seconds for the transfer to initiate and then query the target table to verify the 2 rows from our 'Hive' source have landed.

In [ ]:
import time
from google.cloud import bigquery

print("Waiting 30s for data transfer to process...")
time.sleep(30)

bq_client = bigquery.Client(project=project_id)
query = f"SELECT * FROM `{project_id}.legacy_migration_target.hive_table`"
try:
    df = bq_client.query(query).to_dataframe()
    print(f"\nSuccess! Found {len(df)} rows in the migrated table:")
    display(df.head())
except Exception as e:
    print(f"\nNote: Data might still be in transit. (Details: {e})")

### 9. Summary of Results
- **Active Migration**: The agent didn't just 'prepare' code; it actively triggered the DTS manual run to move data into BigQuery.
- **No custom glue code**: Managed MCP servers handle the hard parts — SQL translation, DDL generation, and transfer scheduling.
- **End-to-end Verification**: We verified the data arrival directly in the notebook, proving the workflow's success.
- **Availability**: Preview as of March 24-25, 2026.